### N-gram language models or how to write scientific papers (4 pts)

We shall train our language model on a corpora of [ArXiv](http://arxiv.org/) articles and see if we can generate a new one!

![img](../resources/monkey.jpg)

_data by neelshah18 from [here](https://www.kaggle.com/neelshah18/arxivdataset/)_

_Disclaimer: this has nothing to do with actual science. But it's fun, so who cares?!_

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

In [3]:
# Alternative manual download link: https://yadi.sk/d/_nGyU2IajjR9-w
# !wget "https://www.dropbox.com/s/99az9n1b57qkd9j/arxivData.json.tar.gz?dl=1" -O arxivData.json.tar.gz
!curl -L -o arxivData.json.tar.gz "https://www.dropbox.com/s/99az9n1b57qkd9j/arxivData.json.tar.gz?dl=1" 
!tar -xvzf arxivData.json.tar.gz
data = pd.read_json("./arxivData.json")
data.sample(n=5)

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   138  100   138    0     0    178      0 --:--:-- --:--:-- --:--:--   178
100    17  100    17    0     0      9      0  0:00:01  0:00:01 --:--:--   204
100   475    0   475    0     0    159      0 --:--:--  0:00:02 --:--:--     0
100 18.0M  100 18.0M    0     0  3227k      0  0:00:05  0:00:05 --:--:-- 9609k
x arxivData.json


,author,day,id,link,month,summary,tag,title,year
14663,"[{'name': 'Nisha Yadav'}, {'name': 'Hrishikesh...",20,0901.3017v1,"[{'rel': 'related', 'href': 'http://dx.doi.org...",1,The Indus script is one of the major undeciphe...,"[{'term': 'cs.CL', 'scheme': 'http://arxiv.org...",Statistical analysis of the Indus script using...,2009
22504,"[{'name': 'Lu Tian'}, {'name': 'Quanquan Gu'}]",15,1610.04798v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",10,We propose a communication-efficient distribut...,"[{'term': 'stat.ML', 'scheme': 'http://arxiv.o...",Communication-efficient Distributed Sparse Lin...,2016
38559,"[{'name': 'Vincent Michel'}, {'name': 'Alexand...",5,1102.1101v1,"[{'rel': 'related', 'href': 'http://dx.doi.org...",2,While medical imaging typically provides massi...,"[{'term': 'cs.CV', 'scheme': 'http://arxiv.org...",Total variation regularization for fMRI-based ...,2011
26361,"[{'name': 'Limin Wang'}, {'name': 'Sheng Guo'}...",4,1610.01119v2,"[{'rel': 'related', 'href': 'http://dx.doi.org...",10,Convolutional Neural Networks (CNNs) have made...,"[{'term': 'cs.CV', 'scheme': 'http://arxiv.org...",Knowledge Guided Disambiguation for Large-Scal...,2016
24322,"[{'name': 'Mario Muštra'}, {'name': 'Mislav Gr...",1,1310.0305v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",10,Breast tissue segmentation into dense and fat ...,"[{'term': 'cs.CV', 'scheme': 'http://arxiv.org...",Filtering for More Accurate Dense Tissue Segme...,2013


In [4]:
# assemble lines: concatenate title and description
lines = data.apply(lambda row: row['title'] + ' ; ' + row['summary'].replace("\n", ' '), axis=1).tolist()

sorted(lines, key=len)[:3]

['Differential Contrastive Divergence ; This paper has been retracted.',
 'What Does Artificial Life Tell Us About Death? ; Short philosophical essay',
 'P=NP ; We claim to resolve the P=?NP problem via a formal argument for P=NP.']

### Tokenization

You know the dril. The data is messy. Go clean the data. Use WordPunctTokenizer or something.


In [5]:
# Task: convert lines (in-place) into strings of space-separated tokens. Import & use WordPunctTokenizer

from nltk.tokenize import WordPunctTokenizer
tokenizer = WordPunctTokenizer()
lines = [' '.join(tokenizer.tokenize(line.lower())) for line in lines]

In [6]:
assert sorted(lines, key=len)[0] == \
    'differential contrastive divergence ; this paper has been retracted .'
assert sorted(lines, key=len)[2] == \
    'p = np ; we claim to resolve the p =? np problem via a formal argument for p = np .'

### N-Gram Language Model (1point)

A language model is a probabilistic model that estimates text probability: the joint probability of all tokens $w_t$ in text $X$: $P(X) = P(w_1, \dots, w_T)$.

It can do so by following the chain rule:

$$
    P(w_1, \dots, w_T) = P(w_1)P(w_2 \mid w_1)\dots P(w_T \mid w_1, \dots, w_{T-1})
$$ 

The problem with such approach is that the final term $P(w_T \mid w_1, \dots, w_{T-1})$ depends on $n-1$ previous words. This probability is impractical to estimate for long texts, e.g. $T = 1000$.

One popular approximation is to assume that next word only depends on a finite amount of previous words:

$$
    P(w_t \mid w_1, \dots, w_{t - 1}) = P(w_t \mid w_{t - n + 1}, \dots, w_{t - 1})
$$

Such model is called __n-gram language model__ where n is a parameter. For example, in 3-gram language model, each word only depends on 2 previous words. 

$$
    P(w_1, \dots, w_n) = \prod_t P(w_t \mid w_{t - n + 1}, \dots, w_{t - 1}).
$$

You can also sometimes see such approximation under the name of _n-th order markov assumption_.

The first stage to building such a model is counting all word occurences given N-1 previous words

In [7]:
from tqdm import tqdm
from collections import defaultdict, Counter

# special tokens:
# - `UNK` represents absent tokens,
# - `EOS` is a special token after the end of sequence

UNK, EOS = "_UNK_", "_EOS_"

def count_ngrams(lines, n):
    """
    Count how many times each word occured after (n - 1) previous words
    :param lines: an iterable of strings with space-separated tokens
    :returns: a dictionary { tuple(prefix_tokens): {next_token_1: count_1, next_token_2: count_2}}

    When building counts, please consider the following two edge cases:
    - every prefix must contain exactly (n - 1) tokens. If a prefix is shorter than that,
      pad it with UNK on the left. The number of padding tokens therefore depends on n:
      derive it from n, do not hard-code it. For n=3,
      empty prefix: "" -> (UNK, UNK)
      short prefix: "the" -> (UNK, the)
      long prefix: "the new approach" -> (new, approach)
      For n=2 the very same prefixes give (UNK,), (the,) and (approach,), and for n=1 all
      of them give the empty tuple (). Equivalently: prepend (n - 1) UNK tokens to a line,
      then slide a window of size n over it.
    - you should add a special token, EOS, at the end of each sequence
      "... with deep neural networks ." -> (..., with, deep, neural, networks, ., EOS)
      count the probability of this token just like all others.

    Useful invariant: a line with L tokens always contributes exactly L + 1 counted events
    (its L tokens plus EOS), for every n. Note also that NGramLanguageModel below already
    implements this padding rule in get_possible_next_tokens - keep the two consistent, or
    the model will look up prefixes that you never counted.
    """
    counts = defaultdict(Counter)
    # counts[(word1, word2)][word3] = how many times word3 occured after (word1, word2)

    for line in lines:
        tokens = [UNK] * (n-1) + line.split() + [EOS]
        for i in range(len(tokens) - n + 1):
            prefix = tuple(tokens[i:i+n-1])
            target_token = tokens[i+n-1]
            counts[prefix].update([target_token])

    return counts


In [8]:
# let's test it
dummy_lines = sorted(lines, key=len)[:100]
dummy_counts = count_ngrams(dummy_lines, n=3)
assert set(map(len, dummy_counts.keys())) == {2}, "please only count {n-1}-grams"
assert len(dummy_counts[('_UNK_', '_UNK_')]) == 78
assert dummy_counts['_UNK_', 'a']['note'] == 3
assert dummy_counts['p', '=']['np'] == 2
assert dummy_counts['author', '.']['_EOS_'] == 1

# the padding rule must hold for every n, not just for n=3
for n in (1, 2, 3, 5, 10):
    counts_n = count_ngrams(dummy_lines, n=n)
    assert set(map(len, counts_n.keys())) == {n - 1}, \
        "every prefix must have exactly (n - 1) tokens, got lengths %s for n=%i" % (
            set(map(len, counts_n.keys())), n)
    assert sum(sum(c.values()) for c in counts_n.values()) == 2877, \
        "each line must contribute exactly len(tokens) + 1 events (its tokens plus EOS) " \
        "for every n. If this passes for n=3 but fails otherwise, you have most likely " \
        "hard-coded the number of UNK padding tokens instead of deriving it from n."

Once we can count N-grams, we can build a probabilistic language model.
The simplest way to compute probabilities is in proporiton to counts:

$$ P(w_t | prefix) = { Count(prefix, w_t) \over \sum_{\hat w} Count(prefix, \hat w) } $$

In [9]:
class NGramLanguageModel:
    def __init__(self, lines, n):
        """
        Train a simple count-based language model:
        compute probabilities P(w_t | prefix) given ngram counts

        :param n: computes probability of next token given (n - 1) previous words
        :param lines: an iterable of strings with space-separated tokens
        """
        assert n >= 1
        assert isinstance(n, int)

        self.n = n

        counts = count_ngrams(lines, self.n)

        # compute token proabilities given counts
        self.probs = defaultdict(Counter)
        # probs[(word1, word2)][word3] = P(word3 | word1, word2)

        # populate self.probs with actual probabilities
        for prefix in counts:
            self.probs[prefix] = counts[prefix].copy()
            total = self.probs[prefix].total()
            for word in self.probs[prefix]:
                self.probs[prefix][word] /= total

    def get_possible_next_tokens(self, prefix):
        """
        :param prefix: string with space-separated prefix tokens
        :returns: a dictionary {token : it's probability} for all tokens with positive probabilities
        """
        prefix = prefix.split()
        prefix = prefix[max(0, len(prefix) - self.n + 1):]
        prefix = [ UNK ] * (self.n - 1 - len(prefix)) + prefix
        return self.probs[tuple(prefix)]

    def get_next_token_prob(self, prefix, next_token):
        """
        :param prefix: string with space-separated prefix tokens
        :param next_token: the next token to predict probability for
        :returns: P(next_token|prefix) a single number, 0 <= P <= 1
        """
        return self.get_possible_next_tokens(prefix).get(next_token, 0)

Let's test it!

In [10]:
dummy_lm = NGramLanguageModel(dummy_lines, n=3)

p_initial = dummy_lm.get_possible_next_tokens('') # '' -> ['_UNK_', '_UNK_']
assert np.allclose(p_initial['learning'], 0.02)
assert np.allclose(p_initial['a'], 0.13)
assert np.allclose(p_initial.get('meow', 0), 0)
assert np.allclose(sum(p_initial.values()), 1)

p_a = dummy_lm.get_possible_next_tokens('a') # '' -> ['_UNK_', 'a']
assert np.allclose(p_a['machine'], 0.15384615)
assert np.allclose(p_a['note'], 0.23076923)
assert np.allclose(p_a.get('the', 0), 0)
assert np.allclose(sum(p_a.values()), 1)

assert np.allclose(dummy_lm.get_possible_next_tokens('a note')['on'], 1)
assert dummy_lm.get_possible_next_tokens('a machine') == \
    dummy_lm.get_possible_next_tokens("there have always been ghosts in a machine"), \
    "your 3-gram model should only depend on 2 previous words"

Now that you've got a working n-gram language model, let's see what sequences it can generate. But first, let's train it on the whole dataset.

In [11]:
lm = NGramLanguageModel(lines, n=3)

The process of generating sequences is... well, it's sequential. You maintain a list of tokens and iteratively add next token by sampling with probabilities.

$ X = [] $

__forever:__
* $w_{next} \sim P(w_{next} | X)$
* $X = concat(X, w_{next})$


Instead of sampling with probabilities, one can also try always taking most likely token, sampling among top-K most likely tokens or sampling with temperature. In the latter case (temperature), one samples from

$$w_{next} \sim {P(w_{next} | X) ^ {1 / \tau} \over \sum_{\hat w} P(\hat w | X) ^ {1 / \tau}}$$

Where $\tau > 0$ is model temperature. If $\tau << 1$, more likely tokens will be sampled with even higher probability while less likely tokens will vanish.

In [12]:
def get_next_token(lm, prefix, temperature=1.0):
    """
    return next token after prefix;
    :param temperature: samples proportionally to lm probabilities ^ (1 / temperature)
        if temperature == 0, always takes most likely token. Break ties arbitrarily.
    """

    dist = lm.get_possible_next_tokens(prefix)
    words = sorted(dist)
    probs = np.array([dist[word] for word in words])

    if temperature > 0.0:
        probs = probs ** (1/temperature)
        probs = probs / probs.sum()
        word = np.random.choice(words, p = probs)

    else:
        word = words[np.argmax(probs)]

    return word
        

In [13]:
from collections import Counter
test_freqs = Counter([get_next_token(lm, 'there have') for _ in range(10000)])
assert 250 < test_freqs['not'] < 450
assert 8500 < test_freqs['been'] < 9500
assert 1 < test_freqs['lately'] < 200

test_freqs = Counter([get_next_token(lm, 'deep', temperature=1.0) for _ in range(10000)])
assert 1500 < test_freqs['learning'] < 3000
test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.5) for _ in range(10000)])
assert 8000 < test_freqs['learning'] < 9000
test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.0) for _ in range(10000)])
assert test_freqs['learning'] == 10000

print("Looks nice!")

Looks nice!


Let's have fun with this model

In [14]:
prefix = 'artificial' # <- your ideas :)

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break

print(prefix)

artificial prediction markets often outperform the state - of - the - art results on english using numerous language resources are optimally detected by reasoning about actions and attributes of entities and the physiology of the data - efficiency but reduced computational complexity in np chunking , named sa - siam is composed by 20 %, and error models for learning in low training error and the extensive experimental analyses , we further leverage the high dimensional data . we show that the information exchange is suppressed . the motion dependencies in such scenarios due to its solution or source term


In [15]:
prefix = 'bridging the' # <- more of your ideas

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix, temperature=0.5)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break

print(prefix)

bridging the gap between the two widely used for the analysis of the target language . in this paper , we propose a new method for the development of a set of selected organisms also move towards the desired output . the proposed approach is based on the relationship between the two - step - size vector ( force ) field based probabilistic approach to the problem of learning a low - rank matrix recovery from rank - 1 accuracy on a number of different data sets . _EOS_


__More in the homework:__ nucleus sampling, top-k sampling, beam search(not for the faint of heart).

### Evaluating language models: perplexity (1point)

Perplexity is a measure of how well your model approximates the true probability distribution behind the data. __Smaller perplexity = better model__.

To compute perplexity on one sentence, use:
$$
    {\mathbb{P}}(w_1 \dots w_N) = P(w_1, \dots, w_N)^{-\frac1N} = \left( \prod_t P(w_t \mid w_{t - n}, \dots, w_{t - 1})\right)^{-\frac1N},
$$


On the corpora level, perplexity is a product of probabilities of all tokens in all sentences to the power of $-\frac1N$, where $N$ is the __total number of scored tokens__ in the corpora.

A sentence $w_1 \dots w_L$ contributes $L + 1$ factors to that product, because EOS is predicted and scored exactly like a regular token:

$$P(w_1 \mid \varnothing) \cdot P(w_2 \mid w_1) \cdots P(w_L \mid w_1 \dots w_{L-1}) \cdot P(EOS \mid w_1 \dots w_L)$$

Here $\varnothing$ is the empty prefix. EOS is counted in $N$ as well, so $N = \sum_{\text{sentences}} (L + 1)$ -- the same number of events you counted in `count_ngrams`.

This number can quickly get too small for float32/float64 precision, so we recommend you to first compute log-perplexity (from log-probabilities) and then take the exponent.

In [16]:
from tqdm.auto import tqdm

def perplexity(lm, lines, min_logprob=np.log(10 ** -50.)):
    """
    :param lines: a list of strings with space-separated tokens
    :param min_logprob: if log(P(w | ...)) is smaller than min_logprop, set it equal to min_logrob
    :returns: corpora-level perplexity - a single scalar number from the formula above

    Note: scoring a line "w_1 w_2 ... w_L" means scoring L + 1 events:
        P(w_1 | ""), P(w_2 | "w_1"), ..., P(w_L | "w_1 ... w_{L-1}"), P(EOS | "w_1 ... w_L")
    The prefix of the very first token is the empty string "", i.e. an empty prefix and not
    a string of UNKs: the model pads it to (n - 1) UNK tokens on its own inside
    get_possible_next_tokens, so do not add any padding here. Likewise, w_1 is simply the
    first real token of the line. The trailing EOS is scored like any other token and is
    counted in N too, so a line with L tokens adds L + 1 to N.

    PLEASE USE lm.get_next_token_prob and NOT lm.get_possible_next_tokens
    """
    denum = 0
    total_log_prob = 0.0
    
    for line in tqdm(lines):
        tokens = line.split() + [EOS]
        prefix = ''
        for token in tokens:
            p = lm.get_next_token_prob(prefix, token)
            logp = max(np.log(p), min_logprob)

            total_log_prob -= logp
            denum += 1

            prefix = prefix + " " + token if prefix else token 

    cross_entropy = total_log_prob/denum

    return np.exp(cross_entropy)

In [17]:
lm1 = NGramLanguageModel(dummy_lines, n=1)
lm3 = NGramLanguageModel(dummy_lines, n=3)
lm10 = NGramLanguageModel(dummy_lines, n=10)

ppx1 = perplexity(lm1, dummy_lines)
ppx3 = perplexity(lm3, dummy_lines)
ppx10 = perplexity(lm10, dummy_lines)
ppx_missing = perplexity(lm3, ['the jabberwock , with eyes of flame , '])  # thanks, L. Carrol

print("Perplexities: ppx1=%.3f ppx3=%.3f ppx10=%.3f" % (ppx1, ppx3, ppx10))

assert all(0 < ppx < 500 for ppx in (ppx1, ppx3, ppx10)), "perplexity should be non-negative and reasonably small"
assert ppx1 > ppx3 > ppx10, "higher N models should overfit and "
assert np.isfinite(ppx_missing) and ppx_missing > 10 ** 6, "missing words should have large but finite perplexity. " \
    " Make sure you use min_logprob right"
assert np.allclose([ppx1, ppx3, ppx10], (318.2132342216302, 1.5199996213739575, 1.1838145037901249))

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

Perplexities: ppx1=318.213 ppx3=1.520 ppx10=1.184


/var/folders/7v/jsyjn5b135x43hj7h86c1s280000gn/T/ipykernel_8998/3210143260.py:27: RuntimeWarning: divide by zero encountered in log
  logp = max(np.log(p), min_logprob)


Now let's measure the actual perplexity: we'll split the data into train and test and score model on test data only.

In [18]:
from sklearn.model_selection import train_test_split
train_lines, test_lines = train_test_split(lines, test_size=0.25, random_state=42)

for n in (1, 2, 3):
    lm = NGramLanguageModel(n=n, lines=train_lines)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))


  0%|          | 0/10250 [00:00<?, ?it/s]

/var/folders/7v/jsyjn5b135x43hj7h86c1s280000gn/T/ipykernel_8998/3210143260.py:27: RuntimeWarning: divide by zero encountered in log
  logp = max(np.log(p), min_logprob)


N = 1, Perplexity = 1832.23136


  0%|          | 0/10250 [00:00<?, ?it/s]

N = 2, Perplexity = 85653987.28774


  0%|          | 0/10250 [00:00<?, ?it/s]

N = 3, Perplexity = 61999196259043346743296.00000


In [19]:
# whoops, it just blew up :)

### LM Smoothing

The problem with our simple language model is that whenever it encounters an n-gram it has never seen before, it assigns it with the probabilitiy of 0. Every time this happens, perplexity explodes.

To battle this issue, there's a technique called __smoothing__. The core idea is to modify counts in a way that prevents probabilities from getting too low. The simplest algorithm here is Additive smoothing (aka [Lapace smoothing](https://en.wikipedia.org/wiki/Additive_smoothing)):

$$ P(w_t | prefix) = { Count(prefix, w_t) + \delta \over \sum_{\hat w} (Count(prefix, \hat w) + \delta) } $$

If counts for a given prefix are low, additive smoothing will adjust probabilities to a more uniform distribution. Not that the summation in the denominator goes over _all words in the vocabulary_.

Here's an example code we've implemented for you:

In [20]:
class LaplaceLanguageModel(NGramLanguageModel):
    """ this code is an example, no need to change anything """
    def __init__(self, lines, n, delta=1.0):
        self.n = n
        counts = count_ngrams(lines, self.n)
        self.vocab = set(token for token_counts in counts.values() for token in token_counts)
        self.probs = defaultdict(Counter)

        for prefix in counts:
            token_counts = counts[prefix]
            total_count = sum(token_counts.values()) + delta * len(self.vocab)
            self.probs[prefix] = {token: (token_counts[token] + delta) / total_count
                                          for token in token_counts}
    def get_possible_next_tokens(self, prefix):
        token_probs = super().get_possible_next_tokens(prefix)
        missing_prob_total = 1.0 - sum(token_probs.values())
        missing_prob = missing_prob_total / max(1, len(self.vocab) - len(token_probs))
        return {token: token_probs.get(token, missing_prob) for token in self.vocab}

    def get_next_token_prob(self, prefix, next_token):
        token_probs = super().get_possible_next_tokens(prefix)
        if next_token in token_probs:
            return token_probs[next_token]
        else:
            missing_prob_total = 1.0 - sum(token_probs.values())
            missing_prob_total = max(0, missing_prob_total) # prevent rounding errors
            return missing_prob_total / max(1, len(self.vocab) - len(token_probs))


**Disclaimer**: the implementation above assumes all words unknown within a given context to be equally likely, *as well as the words outside of vocabulary*. Therefore, its' perplexity will be lower than it should when encountering such words. Therefore, comparing it with a model with fewer unknown words will not be fair. When implementing your own smoothing, you may handle this by adding a virtual `UNK` token of non-zero probability. Technically, this will result in a model where probabilities do not add up to $1$, but it is close enough for a practice excercise.

In [21]:
#test that it's a valid probability model
for n in (1, 2, 3):
    dummy_lm = LaplaceLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

In [22]:
for n in (1, 2, 3):
    lm = LaplaceLanguageModel(train_lines, n=n, delta=0.1)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

  0%|          | 0/10250 [00:00<?, ?it/s]

N = 1, Perplexity = 977.67559


  0%|          | 0/10250 [00:00<?, ?it/s]

N = 2, Perplexity = 470.48021


  0%|          | 0/10250 [00:00<?, ?it/s]

N = 3, Perplexity = 3679.44765


In [23]:
prefix = 'bridging the' # <- more of your ideas

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix, temperature=0.5)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break

print(prefix)

bridging the jections gallery wmt15 simvastatin obtuse unattributed vedai humidity mfcm bouligand wonderland neuman unranked devised clpfd hajnal yeti conceptually gynäkologischen redress squared anatoliev bioplausible centreline proteome misclassified bkf cloudera 1966 explain voila mimetic 9262 lili empathy amounts 3400 41mm pets wan klopotek ruziziensis bsi analyze brewka departing reusable strikes volves eta_t esport bookmarking briefcase dertinger frommultiply hlstm mappable offshore lccf retinal conduced outlier matroidal exquisitely 711 datase lnex poison technologies rtb gatherer palate summa wargames npc kant yc destinations extrapositional lmcs driu researches devrob hevc thriller perturbed weakness smspk lepski hint lmf nadl internalist 948 dqn gifted unannotated histories gqsi simulink


### Kneser-Ney smoothing (2 points)

Additive smoothing is simple, reasonably good but definitely not a State of The Art algorithm.


Your final task in this notebook is to implement [Kneser-Ney](https://en.wikipedia.org/wiki/Kneser%E2%80%93Ney_smoothing) smoothing.

It can be computed recurrently, for n>1:

$$P_{kn}(w_t | prefix_{n-1}) = { \max(0, Count(prefix_{n-1}, w_t) - \delta) \over \sum_{\hat w} Count(prefix_{n-1}, \hat w)} + \lambda_{prefix_{n-1}} \cdot P_{kn}(w_t | prefix_{n-2})$$

where
- $prefix_{n-1}$ is a tuple of {n-1} previous tokens
- $\lambda_{prefix_{n-1}}$ is a normalization constant chosen so that probabilities add up to 1. Discounting takes $\delta$ away from each of the $N_{1+}(prefix_{n-1}, \bullet)$ __distinct__ tokens ever seen after that prefix, and that is exactly the mass to redistribute:
$$\lambda_{prefix_{n-1}} = {\delta \cdot N_{1+}(prefix_{n-1}, \bullet) \over \sum_{\hat w} Count(prefix_{n-1}, \hat w)}$$
- $P_{kn}(w_t | prefix_{n-2})$ is the lower-order model: Kneser-Ney smoothing for the {n-1}-gram language model, i.e. the same formula with the oldest token dropped from the prefix.
- Unigram $P_{kn}(w_t)$ is a special case that terminates the recursion: how likely it is to see $w_t$ in an unfamiliar context

__Prefixes that were never seen in training.__ If $Count(prefix_{n-1}, \hat w) = 0$ for every $\hat w$, then the denominator is zero and both the first term and $\lambda_{prefix_{n-1}}$ come out as $\tfrac{0}{0}$. There is simply no higher-order evidence to interpolate with, so back off completely: take $\lambda_{prefix_{n-1}} = 1$ and the first term equal to $0$, which means returning $P_{kn}(w_t | prefix_{n-2})$ unchanged. The result is still a valid probability distribution, and this is the case you will hit constantly at test time.

__Choosing $\delta$.__ Keep $\delta \in (0, 1]$. Counts are integers $\ge 1$, so within that range $\max(0, Count - \delta) = Count - \delta$ and the probabilities sum to exactly 1. With $\delta > 1$ the $\max$ clips away less mass than $\lambda$ hands back to the lower order, and the distribution stops normalizing -- the validity test below will fail.

See lecture slides or wiki for more detailed formulae.

__Your task__ is to
- implement `KneserNeyLanguageModel` class,
- test it on 1-3 gram language models
- find optimal (within reason) smoothing delta for 3-gram language model with Kneser-Ney smoothing

_Note: the test below iterates over `dummy_lm.vocab`, so your class has to expose a `self.vocab` set holding every known token, just like `LaplaceLanguageModel` does._

In [30]:
def count_unique_predecessor(lines):

    counts = defaultdict(set)
    # counts[word] = set of unique tokens prev_word across all 2-grams (prev_word, word)

    for line in lines:
        tokens = [UNK] + line.split() + [EOS]
        for i in range(len(tokens) - 1):
            counts[tokens[i+1]].add(tokens[i])

    counts = {key:len(el) for key, el in counts.items()}
    return counts


In [98]:
class KneserNeyLanguageModel(NGramLanguageModel):
    """ A template for Kneser-Ney language model. Default delta may be suboptimal. """
    def __init__(self, lines, n, delta=1.0):
        self.n = n
        self.vocab = set(token for line in lines for token in line.split())
        self.vocab.add(EOS)
        self.delta = delta

        prior_counts = count_unique_predecessor(lines)
        prior_probs = defaultdict(lambda :0.0)

        total_count = sum(prior_counts.values())
        for word in prior_counts:
            prior_probs[word] = prior_counts[word] / total_count


        self.probs = {1:prior_probs}
        self.lambda_weights = {}

        for i in range(2, n+1):

            counts = count_ngrams(lines, i)
            totals = {prefix:counter.total() for prefix, counter in counts.items()}
            unique_suffix_counts = {prefix:len(counter) for prefix, counter in counts.items()}

            
            probs = {}
            for prefix, counter in counts.items():
                probs[prefix] = defaultdict(lambda: 0.0)
                for word, count in counter.items():
                    probs[prefix][word] = max(count-self.delta, 0)/totals[prefix]


            lambda_weights = defaultdict(lambda: 1.0)
            
            for prefix in unique_suffix_counts:
                lambda_weights[prefix] = self.delta * unique_suffix_counts[prefix] / totals[prefix]

            self.probs[i] = probs
            self.lambda_weights[i] = lambda_weights

    def _tokenize_prefix(self, prefix, n):

        prefix = prefix.split()
        prefix = prefix[max(0, len(prefix) - n + 1):]
        prefix = [ UNK ] * (n - 1 - len(prefix)) + prefix

        return tuple(prefix)

    def _get_next_token_prob_recursive(self, prefix_tokens, next_token, i):

        if i == 1:
            return self.probs[1][next_token]

        backoff_prob = self._get_next_token_prob_recursive(prefix_tokens[1:],next_token, i-1)
        
        if prefix_tokens not in self.probs[i]:
            return backoff_prob

        else:
            return self.probs[i][prefix_tokens][next_token] + self.lambda_weights[i][prefix_tokens]*backoff_prob
                
    def get_possible_next_tokens(self, prefix):

        ### generation is kinda slow because every token from vocab has 
        ### nonzero probability 
        ### each token generation takes O(vocab_size)
        
        return {word:self.get_next_token_prob(prefix, word) for word in self.vocab}

    def get_next_token_prob(self, prefix, next_token):
        prefix_tokens = self._tokenize_prefix(prefix, self.n)
        return self._get_next_token_prob_recursive(prefix_tokens, next_token, self.n)

In [92]:
dummy_lm = KneserNeyLanguageModel(dummy_lines, n=2)


In [93]:
#test that it's a valid probability model
for n in (1, 2, 3):
    dummy_lm = KneserNeyLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

In [94]:
for n in (1, 2, 3):
    lm = KneserNeyLanguageModel(train_lines, n=n, delta=0.2)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

  0%|          | 0/10250 [00:00<?, ?it/s]

/var/folders/7v/jsyjn5b135x43hj7h86c1s280000gn/T/ipykernel_8998/3210143260.py:27: RuntimeWarning: divide by zero encountered in log
  logp = max(np.log(p), min_logprob)


N = 1, Perplexity = 2885.69413


  0%|          | 0/10250 [00:00<?, ?it/s]

N = 2, Perplexity = 424.82398


  0%|          | 0/10250 [00:00<?, ?it/s]

N = 3, Perplexity = 463.12892


In [84]:
for n in (1, 2, 3):
    lm = KneserNeyLanguageModel(train_lines, n=n, delta=0.5)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

  0%|          | 0/10250 [00:00<?, ?it/s]

/var/folders/7v/jsyjn5b135x43hj7h86c1s280000gn/T/ipykernel_8998/3210143260.py:27: RuntimeWarning: divide by zero encountered in log
  logp = max(np.log(p), min_logprob)


N = 1, Perplexity = 2885.69413


  0%|          | 0/10250 [00:00<?, ?it/s]

N = 2, Perplexity = 392.38092


  0%|          | 0/10250 [00:00<?, ?it/s]

N = 3, Perplexity = 326.33930


In [86]:
for n in (1, 2, 3):
    lm = KneserNeyLanguageModel(train_lines, n=n, delta=0.8)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

  0%|          | 0/10250 [00:00<?, ?it/s]

/var/folders/7v/jsyjn5b135x43hj7h86c1s280000gn/T/ipykernel_8998/3210143260.py:27: RuntimeWarning: divide by zero encountered in log
  logp = max(np.log(p), min_logprob)


N = 1, Perplexity = 2885.69413


  0%|          | 0/10250 [00:00<?, ?it/s]

N = 2, Perplexity = 385.79491


  0%|          | 0/10250 [00:00<?, ?it/s]

N = 3, Perplexity = 286.54721


In [95]:
### ooops stack more context doesnt help
for n in range(1,6):
    lm = KneserNeyLanguageModel(train_lines, n=n, delta=0.8)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

  0%|          | 0/10250 [00:00<?, ?it/s]

/var/folders/7v/jsyjn5b135x43hj7h86c1s280000gn/T/ipykernel_8998/3210143260.py:27: RuntimeWarning: divide by zero encountered in log
  logp = max(np.log(p), min_logprob)


N = 1, Perplexity = 2885.69413


  0%|          | 0/10250 [00:00<?, ?it/s]

N = 2, Perplexity = 385.79491


...

N = 3, Perplexity = 286.54721


  0%|          | 0/10250 [00:00<?, ?it/s]

N = 4, Perplexity = 290.73608


  0%|          | 0/10250 [00:00<?, ?it/s]

N = 5, Perplexity = 297.84684


In [100]:
lm = KneserNeyLanguageModel(train_lines, n=3, delta=0.8)

prefix = 'self supervised attention' # <- more of your ideas


for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix, temperature=0.2)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break

print(prefix)

self supervised attention network ( cnn ) to the best of our approach is to identify the best of our approach is based on the other hand , we propose a novel approach to the state - of - the - art methods . _EOS_
